<a href="https://colab.research.google.com/github/Marfall/IT-PROJECT-RISK/blob/main/IT_Project_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Управление IT-проектами на основе данных: прогнозирование успеха, интерпретация рисков и предиктивный мониторинг потерь

**Дипломный проект**

---

## 📌 О проекте

Значительная часть IT-проектов не достигает поставленных целей: сроки сдвигаются, бюджет превышается, продукт не приносит ожидаемой ценности. Как правило, причины становятся очевидны уже после завершения, когда влиять на исход поздно.

Цель проекта — разработать систему поддержки управленческих решений, которая позволяет:
- прогнозировать вероятность успеха проекта на ранней стадии;
- выявлять и интерпретировать ключевые факторы риска;
- отслеживать динамику риска во времени;
- формировать понятные рекомендации для руководителя.

Фактически, это инструмент, который переводит управление проектами с интуитивного уровня на уровень, основанный на данных.

---

## 🧩 Структура системы

| Модуль | Назначение |
|--------|------------|
| **LightGBM + SHAP** | Прогноз риска и объяснение причин |
| **NLP (sentence-transformers)** | Анализ текстов требований |
| **Prophet** | Мониторинг динамики риска по неделям |
| **MLP (PyTorch)** | Сравнение с бустингом для обоснования выбора |
| **Seaborn + Plotly** | Визуализация, ориентированная на бизнес |

---

## 🛠️ Технологический стек

- **Python** — язык реализации
- **pandas, numpy, sklearn** — обработка данных и пайплайны
- **LightGBM** — основная модель
- **SHAP** — интерпретация модели
- **sentence-transformers** — работа с текстом
- **Prophet** — временные ряды
- **PyTorch** — нейросеть для сравнения
- **Seaborn, Plotly** — визуализация
- **Jupyter Notebook** — среда разработки

---

## 📊 Источники данных

- **Project Management Risk Raw** (Kaggle) — 4000 проектов, 51 признак
- **FR_NFR_dataset** (Mendeley) — 6118 требований для NLP-модуля
- **Временной ряд** — синтетический, поскольку в исходных данных отсутствуют временные метки

---

## 📖 Структура работы

1. Введение
2. Данные и EDA
3. Предобработка и пайплайн
4. Прогнозирование успеха (LightGBM)
5. Интерпретация рисков (SHAP)
6. NLP-модуль
7. Временной мониторинг (Prophet)
8. Сравнение с нейросетью
9. Дашборд
10. Выводы и рекомендации

---

## 💰 Бизнес-ценность

- **Прогноз:** проект зелёный / жёлтый / красный
- **Причины:** факторы, которые повышают риск
- **Динамика:** изменение риска по неделям
- **Рекомендации:** управленческие действия для снижения потерь

---

*Автор: [Весновский Александр Андреевич]*  
*Курс: ML Pro, OTUS*  
*Дата: [21.09.2026]*

## 1. Импорт библиотек и настройка окружения

Подключаем базовый стек для анализа данных и визуализации:
- `pandas`, `numpy` — работа с таблицами и массивами;
- `matplotlib`, `seaborn` — графики;
- настраиваем единый стиль визуализации и отображение всех колонок датасета.

Фиксируем seed, чтобы результаты были воспроизводимы при повторных запусках.

In [4]:
# 2: Импорт библиотек, настройка стиля и фиксация seed
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Стиль графиков
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Отображение всех колонок в выводе DataFrame
pd.set_option("display.max_columns", None)

# Фиксация seed для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Seed зафиксирован: {SEED}")

Seed зафиксирован: 42


## 3. Загрузка датасета

На этом шаге подключаем основной датасет — **Project Management Risk Raw** (Kaggle). Датасет содержит 4000 IT-проектов и 51 признак, включая целевую переменную `Risk_Level`.

Логика загрузки:
1. Проверяем наличие файла в стандартных местах (рабочая директория, Google Drive, текущая папка).
2. Если файла нет — скачиваем через Kaggle API.
3. Если API недоступен — оставляем инструкцию для ручной загрузки.

На выходе получаем переменную `DATASET_PATH` с путём к файлу, которую используют следующие ячейки.

In [5]:
# 4: Загрузка датасета Project Management Risk Raw из Google Drive
import os
from pathlib import Path

# Монтируем Google Drive (если ещё не смонтирован)
from google.colab import drive
drive.mount('/content/drive')

# Папка проекта на Drive и имя файла датасета
PROJECT_DIR = Path("/content/drive/MyDrive/ML-PROJECT-RISK-MANAGEMENT-DIPLOMA")
KAGGLE_FILE = "project_risk_raw_dataset.csv"

# Ищем файл датасета в папке проекта
DATASET_PATH = None
if PROJECT_DIR.exists():
    matches = list(PROJECT_DIR.rglob(KAGGLE_FILE))
    if matches:
        DATASET_PATH = str(matches[0])
        print(f"Файл найден: {DATASET_PATH}")
    else:
        print(f"В папке {PROJECT_DIR} файл {KAGGLE_FILE} не найден.")
        print("Содержимое папки:")
        for p in PROJECT_DIR.iterdir():
            print(f"  {p.name}")
else:
    print(f"Папка {PROJECT_DIR} не найдена. Проверьте путь на Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Файл найден: /content/drive/MyDrive/ML-PROJECT-RISK-MANAGEMENT-DIPLOMA/project_risk_raw_dataset.csv


## 5. Чтение датасета и первый обзор

Читаем CSV в DataFrame и смотрим базовую структуру:
- `head()` — как выглядят первые строки, какие есть колонки;
- `shape` — размер таблицы (строки × столбцы);
- `info()` — типы данных и количество непустых значений (сразу видно, где пропуски);
- `describe()` — основные статистики по числовым признакам: среднее, минимум, максимум, квартили.

На этом шаге цель — понять, с чем вообще работаем, до любых преобразований.

In [8]:
# 6: Чтение CSV и первичный обзор структуры данных
df = pd.read_csv(DATASET_PATH)

# Первые строки
print("Первые 5 строк:")
display(df.head())

# Размер таблицы
print(f"\nРазмер: {df.shape[0]} строк, {df.shape[1]} колонок")

# Типы данных и пропуски — первые 15 колонок, чтобы не заваливать вывод
print("\nСтруктура (первые 15 колонок):")
df.iloc[:, :15].info()

# Описательные статистики — первые 15 колонок
print("\nОписательные статистики (первые 15 колонок):")
display(df.iloc[:, :15].describe().T)

Первые 5 строк:


,Project_ID,Project_Type,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Methodology_Used,Team_Experience_Level,Past_Similar_Projects,External_Dependencies_Count,Change_Request_Frequency,Project_Phase,Requirement_Stability,Team_Turnover_Rate,Vendor_Reliability_Score,Historical_Risk_Incidents,Communication_Frequency,Regulatory_Compliance_Level,Technology_Familiarity,Geographical_Distribution,Stakeholder_Engagement_Level,Schedule_Pressure,Budget_Utilization_Rate,Executive_Sponsorship,Funding_Source,Market_Volatility,Integration_Complexity,Resource_Availability,Priority_Level,Organizational_Change_Frequency,Cross_Functional_Dependencies,Previous_Delivery_Success_Rate,Technical_Debt_Level,Project_Manager_Experience,Org_Process_Maturity,Data_Security_Requirements,Key_Stakeholder_Availability,Tech_Environment_Stability,Contract_Type,Resource_Contention_Level,Industry_Volatility,Client_Experience_Level,Change_Control_Maturity,Risk_Management_Maturity,Team_Colocation,Documentation_Quality,Project_Start_Month,Current_Phase_Duration_Months,Seasonal_Risk_Factor,Risk_Level
0,PROJ_0001,Construction,32,1526276.55,32,9.70,16,Waterfall,Senior,3,3,1.05,Planning,Moderate,0.16,0.84,2,1.90,Medium,Expert,4,Medium,0.0,0.82,Moderate,Government,0.55,2.66,0.98,Medium,0.29,6,0.80,0.00,Mid-level PM,Managed,Medium,Limited,NaN,Time & Materials,High,Extreme,First-time,Basic,Basic,Fully Colocated,Good,10,5,1.0,High
1,PROJ_0002,Manufacturing,2,390790.15,9,2.72,9,Kanban,Mixed,0,2,2.61,Execution,Moderate,0.42,0.79,2,2.65,High,Familiar,5,Excellent,0.0,0.76,Weak,External,0.29,2.45,0.95,Low,0.50,3,0.73,0.00,Mid-level PM,Optimizing,Low,Excellent,NaN,Cost-Plus,Low,Stable,Occasional,Advanced,Formal,Fully Remote,Poor,9,3,1.0,Low
2,PROJ_0003,Manufacturing,2,246674.76,6,2.04,7,Agile,Mixed,1,0,0.83,Initiation,Stable,0.55,0.89,2,1.87,Medium,Familiar,3,Excellent,0.0,1.17,Strong,Internal,0.15,5.57,0.79,High,2.71,2,0.91,0.00,Mid-level PM,Ad-hoc,Medium,Moderate,NaN,Cost-Plus,High,Stable,Regular,NaN,NaN,Hybrid,Good,5,1,1.0,Medium
3,PROJ_0004,IT,12,1427830.63,17,7.54,16,Scrum,Mixed,0,5,2.42,Initiation,Moderate,0.33,0.84,1,1.36,Low,Familiar,5,Medium,0.1,0.73,Moderate,Mixed,0.74,7.49,0.52,Critical,0.74,5,0.71,0.43,Certified PM,Managed,Strict,Good,Mixed,Hybrid,High,Extreme,Strategic,Formal,Basic,Hybrid,Basic,12,6,1.1,High
4,PROJ_0005,Construction,24,1696746.64,24,6.68,17,Hybrid,Junior,0,2,0.16,Execution,Stable,0.36,0.86,1,1.02,Critical,Expert,2,Low,0.0,0.60,Moderate,Government,0.19,1.64,0.58,Medium,0.56,6,0.83,0.00,Certified PM,Defined,Medium,Moderate,NaN,Cost-Plus,High,Moderate,Occasional,Basic,NaN,Partially Colocated,Basic,9,6,1.0,High



Размер: 4000 строк, 51 колонок

Структура (первые 15 колонок):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Project_ID                   4000 non-null   object 
 1   Project_Type                 4000 non-null   object 
 2   Team_Size                    4000 non-null   int64  
 3   Project_Budget_USD           4000 non-null   float64
 4   Estimated_Timeline_Months    4000 non-null   int64  
 5   Complexity_Score             4000 non-null   float64
 6   Stakeholder_Count            4000 non-null   int64  
 7   Methodology_Used             4000 non-null   object 
 8   Team_Experience_Level        4000 non-null   object 
 9   Past_Similar_Projects        4000 non-null   int64  
 10  External_Dependencies_Count  4000 non-null   int64  
 11  Change_Request_Frequency     4000 non-null   float64
 12  Project_Phas

,count,mean,std,min,25%,50%,75%,max
Team_Size,4000.0,1.538825e+01,9.220969,2.00,9.000,13.000,2.000000e+01,50.00
Project_Budget_USD,4000.0,1.143032e+06,590878.115350,159355.55,692532.915,1007471.805,1.475870e+06,3768354.37
Estimated_Timeline_Months,4000.0,1.714775e+01,6.926609,2.00,12.000,17.000,2.200000e+01,36.00
Complexity_Score,4000.0,6.192525e+00,2.212538,1.62,4.460,6.015,7.862500e+00,10.00
Stakeholder_Count,4000.0,1.113050e+01,4.425875,2.00,8.000,10.000,1.400000e+01,29.00
Past_Similar_Projects,4000.0,1.973750e+00,1.750093,0.00,1.000,2.000,3.000000e+00,10.00
External_Dependencies_Count,4000.0,3.127750e+00,1.609216,0.00,2.000,3.000,4.000000e+00,7.00
Change_Request_Frequency,4000.0,1.638080e+00,1.170451,0.01,0.760,1.370,2.230000e+00,8.84
Team_Turnover_Rate,4000.0,2.927250e-01,0.166546,0.00,0.160,0.270,4.000000e-01,0.85


## 7. Первичный обзор данных

**Что видим:**
- 4000 строк, 51 колонка — достаточно для обучения, но без избытка.
- Целевая переменная — `Risk_Level` (High / Medium / Low). Задача — классификация.
- Признаки смешанные: числовые (бюджет, сроки, сложность), категориальные (тип проекта, методология, фаза), порядковые (уровень опыта).
- В первых 15 колонках пропусков нет. В остальных (например, `Tech_Environment_Stability`) встречаются NaN — проверим отдельно.
- `Project_ID` — уникальный идентификатор, для модели не нужен. Удалим.

**Что это значит для проекта:**
- Данные подходят под задачу: есть цель и богатый набор признаков.
- Категориальные признаки потребуют кодирования (One-Hot или Target Encoding).
- Числовые признаки в разном масштабе. Для LightGBM масштабирование не критично, для MLP — обязательно.
- Пропуски есть, но их немного — обработаем на этапе предобработки.

**Следующий шаг:**
1. Проверить пропуски по всем 51 колонке.
2. Посмотреть распределение `Risk_Level` — есть ли дисбаланс классов.
3. Проверить дубликаты и аномалии.